# Silver Layer — SCD Type 2 (Yelp Users)

## Logic MERGE INTO
Detect thay đổi bằng cách so sánh `review_count`, `fans`, `average_stars` với bản ghi active hiện tại.

**Khi có thay đổi:**
1. **BƯỚC 1**: Đóng record cũ — set `is_current = false`, `end_time = now()`
2. **BƯỚC 2**: Insert record mới — `is_current = true`, `effective_time = now()`, `end_time = NULL`

**Khi không có thay đổi:** bỏ qua (idempotent)

**Khi user mới (chưa tồn tại):** Insert trực tiếp

> Notebook đã module hóa — toàn bộ logic core nằm trong package `src/`.
> Notebook chỉ setup SparkSession, gọi function, và làm phần interactive (monitor/validate).


In [1]:
from pathlib import Path
import sys

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.spark_session import get_spark_session

spark = get_spark_session("Yelp_Silver_SCD2")
print("✅ SparkSession ready")

✅ SparkSession ready


26/06/28 13:21:18 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
from src.silver import create_silver_table

# Stop stream cũ
for q in spark.streams.active:
    q.stop()
    print(f"⏹ Stopped: {q.name}")

create_silver_table(spark)
print("✅ Table nessie.silver.yelp_users_scd2 ready")

✅ Table nessie.silver.yelp_users_scd2 ready


In [3]:
from src.silver import start_silver_stream

print("[*] Khởi chạy Silver SCD2 stream...")
silver_query = start_silver_stream(spark, trigger_seconds=20)
print("[+] Silver SCD2 stream đang chạy!")

[*] Khởi chạy Silver SCD2 stream...
[+] Silver SCD2 stream đang chạy!


In [11]:
# Monitor SCD2 — xem tỷ lệ active vs expired
import time
from src.silver import get_scd2_stats

for i in range(1):
    try:
        stats = get_scd2_stats(spark)
        total, active_n, expired = stats["total_records"], stats["active_records"], stats["expired_records"]
        streams = len(spark.streams.active)
        print(f"[{i+1:02d}] total={total:>10,} | active={active_n:>10,} | expired={expired:>8,} | streams={streams}")
    except Exception as e:
        print(f"[{i+1:02d}] Error: {e}")
    time.sleep(20)

[01] total= 2,783,055 | active= 1,987,897 | expired= 795,158 | streams=1


In [12]:
# ============================================================
# VALIDATION — Chứng minh SCD2 hoạt động
# ============================================================
from src.silver import get_users_with_history

print("=" * 60)
print("VALIDATION: Users có lịch sử thay đổi (version > 1)")
print("=" * 60)

changed = get_users_with_history(spark, min_versions=2, limit=10)
changed.show(truncate=False)
count = changed.count()
print(f"→ Tìm thấy {count:,} users có lịch sử thay đổi")

if count > 0:
    print("\n✅ SCD Type 2 hoạt động đúng — có UPDATE history!")
else:
    print("\n⚠️  Chưa có history — Silver stream chưa xử lý xong Pass 2/3")

VALIDATION: Users có lịch sử thay đổi (version > 1)


+----------------------+-------------+--------------------------+--------------------------+
|user_id               |version_count|first_seen                |last_changed              |
+----------------------+-------------+--------------------------+--------------------------+
|0wC6X-PmIdCPWlP2e0DQGw|3            |2026-06-28 13:22:02.17977 |2026-06-28 13:54:23.526229|
|0V78YnDChv4e8L4zmMWzrQ|3            |2026-06-28 13:21:28.388902|2026-06-28 13:54:23.526229|
|-2PTRQF3NwulQlTtVhr_qA|3            |2026-06-28 13:21:28.388902|2026-06-28 13:54:23.526229|
|4ACw6YewzazglfOmZArppg|3            |2026-06-28 13:21:43.559898|2026-06-28 13:54:23.526229|
|-d5RO4FJRtornnq88122sw|3            |2026-06-28 13:21:28.388902|2026-06-28 13:54:23.526229|
|4pqQK5X6nJ3D0OaGybanXQ|3            |2026-06-28 13:21:28.388902|2026-06-28 13:54:23.526229|
|2CnoQq7UidyiPMQ-7sGBrw|3            |2026-06-28 13:21:28.388902|2026-06-28 13:54:23.526229|
|7nGxoSz20iY7CxBMJrkLKw|3            |2026-06-28 13:21:43.559898|2026-

→ Tìm thấy 10 users có lịch sử thay đổi

✅ SCD Type 2 hoạt động đúng — có UPDATE history!


In [13]:
# Full history của 1 user cụ thể (chọn user có nhiều version nhất)
from src import config

top_user = spark.sql(f"""
    SELECT user_id, COUNT(*) as v
    FROM {config.TABLE_SILVER_USERS_SCD2}
    GROUP BY user_id
    ORDER BY v DESC
    LIMIT 1
""").collect()[0]["user_id"]

print(f"User có nhiều version nhất: {top_user}")
print()

spark.sql(f"""
    SELECT
        user_id, review_count, fans, average_stars,
        is_current, effective_time, end_time
    FROM {config.TABLE_SILVER_USERS_SCD2}
    WHERE user_id = '{top_user}'
    ORDER BY effective_time ASC
""").show(truncate=False)

User có nhiều version nhất: -RQJhkPHEcIeLlKfCdV8CA

+----------------------+------------+----+-------------+----------+--------------------------+--------------------------+
|user_id               |review_count|fans|average_stars|is_current|effective_time            |end_time                  |
+----------------------+------------+----+-------------+----------+--------------------------+--------------------------+
|-RQJhkPHEcIeLlKfCdV8CA|3           |0   |4.67         |false     |2026-06-28 13:21:28.388902|2026-06-28 13:47:20.18442 |
|-RQJhkPHEcIeLlKfCdV8CA|12          |3   |4.6          |false     |2026-06-28 13:47:28.471905|2026-06-28 13:51:00.208015|
|-RQJhkPHEcIeLlKfCdV8CA|26          |6   |4.75         |true      |2026-06-28 13:51:03.246235|NULL                      |
+----------------------+------------+----+-------------+----------+--------------------------+--------------------------+



In [14]:
# Iceberg Time Travel — xem snapshot tại thời điểm Pass 1 (chỉ có initial load)
from src.iceberg_utils import get_snapshot_history
from src import config

snaps = get_snapshot_history(spark, config.TABLE_SILVER_USERS_SCD2, exclude_compaction=False)
for s in snaps:
    print(s["snapshot_id"], s["committed_at"], s["operation"])

if len(snaps) >= 2:
    first_snap = snaps[0]["snapshot_id"]
    print(f"\nTime travel → snapshot đầu tiên (Pass 1): {first_snap}")
    count_v1 = spark.sql(f"""
        SELECT COUNT(*) as cnt
        FROM {config.TABLE_SILVER_USERS_SCD2} VERSION AS OF {first_snap}
    """).collect()[0]["cnt"]
    count_now = spark.sql(f"SELECT COUNT(*) as cnt FROM {config.TABLE_SILVER_USERS_SCD2}").collect()[0]["cnt"]
    print(f"  Records tại snapshot 1 : {count_v1:,}")
    print(f"  Records hiện tại       : {count_now:,}")
    print(f"  Tăng thêm              : {count_now - count_v1:,} (đây là SCD2 history records)")

871060912108642657 2026-06-28 13:21:28.183000 overwrite
3427625663138027210 2026-06-28 13:21:39.442000 append
1220817731148145352 2026-06-28 13:21:43.489000 overwrite
5621647349914974691 2026-06-28 13:21:45.607000 append
5938617636349446055 2026-06-28 13:22:02.085000 overwrite
6048040186808781875 2026-06-28 13:22:03.580000 append
7500925027719209762 2026-06-28 13:22:22.178000 overwrite
4066734360648605825 2026-06-28 13:22:24.122000 append
1080079780469978646 2026-06-28 13:22:42.127000 overwrite
7859316330521326339 2026-06-28 13:22:44.431000 append
61351482985055187 2026-06-28 13:24:09.652000 overwrite
537131339461102622 2026-06-28 13:24:12.220000 append
8689990709496053820 2026-06-28 13:24:26.369000 overwrite
3115198450270500937 2026-06-28 13:24:27.547000 append
5266500781575022100 2026-06-28 13:24:46.643000 overwrite
3015528973757458497 2026-06-28 13:24:48.025000 append
1165367553512169492 2026-06-28 13:25:06.989000 overwrite
6121261738514082642 2026-06-28 13:25:08.537000 append
82953